<a href="https://colab.research.google.com/github/hailleenvarela/desafio-motriz-dados/blob/main/01_ingestao_tratamento_validacao_motriz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 01 — Ingestão, tratamento e validação | Case Motriz

**Objetivo:** ler cada fonte separadamente no Google Colab, aplicar as regras R1–R10 do enunciado, gerar tabelas Bronze/Silver em arquivos CSV e os três resultados finais com layout exato. Este notebook **não carrega no BigQuery**; ele prepara dados validados para o Notebook 02.

## Regras importantes
- R1: somente 03/08/2026–30/10/2026 e `dia_letivo = S` no calendário oficial.
- R2: data, aluno, turma, escola e situação devem estar identificados/validados.
- R3: presença conta 1; falta e falta justificada contam 0 no numerador, mas permanecem no denominador; desconhecido é descartado.
- R4: identidade é nome normalizado + nascimento, dentro da rede.
- R5: um registro por aluno/turma/dia letivo; a chave inclui INEP para não confundir turmas iguais em escolas diferentes.
- R6: início/fim de matrícula, quando fornecidos, são inclusivos.
- R7: INEP normalizado para 8 dígitos.
- R8: agregar somando presenças e registros válidos, nunca fazendo média de percentuais.
- R9: lista abaixo de 75% no período completo e em todas as matrículas da pessoa dentro da rede.
- R10: mês deriva da data da aula (`AAAA-MM`).

**Privacidade:** não envie dados brutos nem credenciais ao GitHub. Arquivos no runtime do Colab desaparecem ao reiniciar, a menos que use o Drive.

In [1]:
%pip -q install pandas unidecode

In [2]:
from pathlib import Path
import json, re, hashlib, unicodedata, csv, io, shutil, math
import pandas as pd, zipfile
from IPython.display import display
from google.colab import files

# Se True, os arquivos persistem no Google Drive; se False, ficam no runtime do Colab.
USAR_DRIVE = True
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/motriz_case')
else:
    BASE = Path('/content/motriz_case')

DADOS = BASE / 'dados'
RESULTADOS = BASE / 'resultados'
BRONZE = BASE / 'bronze'
SILVER = BASE / 'silver'
for p in [DADOS/'rede_a', DADOS/'rede_b', DADOS/'rede_c', RESULTADOS, BRONZE, SILVER]:
    p.mkdir(parents=True, exist_ok=True)

DATA_INICIO, DATA_FIM = '2026-08-03', '2026-10-30'
print('Pasta de trabalho:', BASE)

Mounted at /content/drive
Pasta de trabalho: /content/drive/MyDrive/motriz_case


## 1. Enviar o calendário letivo
Envie `calendario_letivo_2026.csv`. O notebook substitui o arquivo de mesmo nome, sem criar cópias numeradas.

In [3]:
uploaded = files.upload()
for name, content in uploaded.items():
    if Path(name).name.lower() == 'calendario_letivo_2026.csv':
        target = DADOS / 'calendario_letivo_2026.csv'
        target.write_bytes(content)
        print('Salvo/substituído:', target)

CAL_PATH = DADOS / 'calendario_letivo_2026.csv'
if CAL_PATH.exists():
    cal = pd.read_csv(CAL_PATH, dtype=str)
    required = {'data', 'dia_letivo'}
    if not required.issubset(cal.columns):
        raise ValueError(f'Calendário sem colunas obrigatórias: {required - set(cal.columns)}')
    cal['data_norm'] = pd.to_datetime(cal['data'], errors='coerce').dt.strftime('%Y-%m-%d')
    if cal['data_norm'].isna().any():
        print('ATENÇÃO: datas inválidas no calendário:', int(cal['data_norm'].isna().sum()))
    cal['dia_letivo_norm'] = cal['dia_letivo'].astype(str).str.strip().str.upper()
    dias_letivos = set(cal.loc[cal['dia_letivo_norm'].eq('S'), 'data_norm'].dropna())
    if not dias_letivos:
        raise ValueError('Nenhum dia letivo S foi encontrado no calendário.')
    print('Dias letivos válidos:', len(dias_letivos))
    display(cal.head())
else:
    raise FileNotFoundError('Envie o calendário antes de tratar as fontes.')

Saving calendario_letivo_2026.csv to calendario_letivo_2026.csv
Salvo/substituído: /content/drive/MyDrive/motriz_case/dados/calendario_letivo_2026.csv
Dias letivos válidos: 63


,data,dia_semana,dia_letivo,observacao,data_norm,dia_letivo_norm
0,2026-08-01,sábado,N,Antes do início do período de apuração,2026-08-01,N
1,2026-08-02,domingo,N,Antes do início do período de apuração,2026-08-02,N
2,2026-08-03,segunda,S,NaN,2026-08-03,S
3,2026-08-04,terça,S,NaN,2026-08-04,S
4,2026-08-05,quarta,S,NaN,2026-08-05,S


## 2. Funções comuns de normalização e qualidade

In [4]:
def norm_text(v):
    if v is None or pd.isna(v): return ''
    s = unicodedata.normalize('NFKD', str(v))
    s = ''.join(ch for ch in s if not unicodedata.combining(ch))
    return re.sub(r'\s+', ' ', s).strip().casefold()

def header_key(v):
    s = norm_text(v).replace('º','').replace('°','')
    return re.sub(r'[^a-z0-9]+', '_', s).strip('_')

def normalize_inep(v):
    if v is None or pd.isna(v): return None
    s = str(v).strip()
    # CSVs/exports podem representar código numérico como 99675471.0
    if re.fullmatch(r'\d+\.0+', s): s = s.split('.')[0]
    digits = re.sub(r'\D', '', s)
    if not digits: return None
    if len(digits) == 7: digits = digits.zfill(8)
    return digits if len(digits) == 8 else None

def parse_date(v):
    """Converte texto ou Extended JSON do Mongo para data ISO; datas inválidas viram None."""
    if isinstance(v, dict):
        if '$date' in v: v = v['$date']
        elif '$numberLong' in v: v = v['$numberLong']
        else: return None
    if v is None or (isinstance(v, float) and math.isnan(v)) or str(v).strip() == '': return None
    # Epoch em milissegundos (Mongo Extended JSON)
    if isinstance(v, (int, float)) or (isinstance(v, str) and re.fullmatch(r'-?\d{12,}', v.strip())):
        try: return pd.to_datetime(int(v), unit='ms', utc=True).date().isoformat()
        except Exception: return None
    # ISO AAAA-MM-DD deve ser interpretado sem dayfirst; outros formatos tentam dia/mês/ano.
    text = str(v).strip()
    dayfirst = not bool(re.match(r'^\d{4}-\d{2}-\d{2}', text))
    dt = pd.to_datetime(v, errors='coerce', utc=True, dayfirst=dayfirst)
    return None if pd.isna(dt) else dt.date().isoformat()

def parse_bool_status(v):
    """Retorna 1 para presença, 0 para falta/F/J e None para situação desconhecida."""
    if v is None or pd.isna(v): return None
    if isinstance(v, bool): return int(v)
    s = norm_text(v)
    if s in {'1','true','t','p','presente','presenca','sim','yes'}: return 1
    if s in {'0','false','f','falta','j','justificada','falta justificada','ausente','nao','no'}: return 0
    return None

def safe_csv(df, path):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding='utf-8', na_rep='')

def save_uploads(uploaded, target_dir, allowed_suffixes):
    """Substitui pelo nome canônico; remove cópias antigas ' (1)' do mesmo nome."""
    target_dir = Path(target_dir); target_dir.mkdir(parents=True, exist_ok=True)
    for original, content in uploaded.items():
        base = Path(original).name
        canonical = re.sub(r' \(\d+\)(?=\.[^.]+$)', '', base)
        if Path(canonical).suffix.lower() not in allowed_suffixes: continue
        target = target_dir / canonical
        target.write_bytes(content)
        pattern = re.compile(r'^' + re.escape(target.stem) + r' \(\d+\)' + re.escape(target.suffix) + r'$', re.I)
        for old in target_dir.iterdir():
            if old.is_file() and pattern.match(old.name): old.unlink()
        print(('Substituído' if target.exists() else 'Salvo'), target.name, f'({len(content):,} bytes)')


## 3. Rede A — dump PostgreSQL em texto
Envie `rede_a_dump.sql`. A extração lê os blocos `COPY` do `pg_dump` sem precisar iniciar um servidor PostgreSQL. As tabelas extraídas são salvas em Bronze para auditoria.

In [5]:
uploaded = files.upload()
save_uploads(uploaded, DADOS/'rede_a', {'.sql'})

def parse_pg_dump(path):
    text = Path(path).read_text(encoding='utf-8', errors='replace')
    # Captura COPY schema.tabela (colunas) FROM stdin; ... \.
    pattern = re.compile(r'COPY\s+educacao\.(\w+)\s+\((.*?)\)\s+FROM stdin;\s*\n(.*?)\n\\\.', re.S | re.I)
    tables = {}
    for match in pattern.finditer(text):
        table, cols, body = match.groups()
        columns = [c.strip().strip('"') for c in cols.split(',')]
        rows = []
        for line in body.splitlines():
            vals = line.split('\t')
            vals = [None if v == r'\N' else v.replace(r'\t','\t').replace(r'\n','\n').replace(r'\r','\r').replace(r'\\','\\') for v in vals]
            if len(vals) != len(columns):
                raise ValueError(f'Linha com {len(vals)} campos, esperado {len(columns)} na tabela {table}')
            rows.append(vals)
        tables[table.lower()] = pd.DataFrame(rows, columns=columns)
    required = {'aluno','escola','frequencia','matricula','turma'}
    missing = required - set(tables)
    if missing: raise ValueError(f'Tabelas obrigatórias ausentes no dump: {sorted(missing)}')
    return tables

path_a = DADOS/'rede_a'/'rede_a_dump.sql'
tabelas_a = parse_pg_dump(path_a) if path_a.exists() else {}
if tabelas_a:
    for name, df in tabelas_a.items():
        safe_csv(df, BRONZE/f'rede_a_{name}.csv')
        print(f'{name}: {len(df):,} linhas; colunas={list(df.columns)}')
    display(tabelas_a['escola'].head())
else:
    print('Dump Rede A ainda não enviado.')

Saving rede_a_dump.sql to rede_a_dump.sql
Substituído rede_a_dump.sql (1,205,858 bytes)
aluno: 417 linhas; colunas=['id', 'nome', 'data_nascimento', 'sexo']
escola: 5 linhas; colunas=['id', 'codigo_inep', 'nome', 'situacao']
frequencia: 26,177 linhas; colunas=['id', 'aluno_id', 'turma_id', 'data_aula', 'presente', 'registrado_em']
matricula: 417 linhas; colunas=['id', 'aluno_id', 'turma_id', 'data_matricula', 'situacao']
turma: 17 linhas; colunas=['id', 'escola_id', 'serie', 'codigo', 'turno', 'ano_letivo']


,id,codigo_inep,nome,situacao
0,1,99158813,Escola Municipal Integral Irmã Dulce Moreira,ATIVA
1,2,99263097,Escola Municipal de Ensino Fundamental Profª J...,ATIVA
2,3,99182761,Escola Municipal Maestro Capiba Neto,ATIVA
3,4,99465873,Escola Municipal Cônego Luís Barreto,ATIVA
4,5,99018898,Escola Municipal de Ensino Fundamental Profª M...,ATIVA


## 4. Rede B — NDJSON
Envie `alunos_rede_b.ndjson`. Expandimos documentos → matrículas → presenças. Dados de responsáveis/telefone não são levados à camada analítica porque não são necessários para o objetivo do case.

In [6]:
uploaded = files.upload()
save_uploads(uploaded, DADOS/'rede_b', {'.ndjson', '.jsonl'})

def extract_b(path):
    rows, errors, anomalies = [], [], []
    docs_read = 0
    with open(path, encoding='utf-8') as f:
        for line_no, line in enumerate(f, 1):
            if not line.strip(): continue
            docs_read += 1
            try: d = json.loads(line)
            except json.JSONDecodeError:
                errors.append({'linha_origem': line_no, 'motivo': 'json_invalido'}); continue
            mats = d.get('matriculas') or []
            if not isinstance(mats, list):
                errors.append({'linha_origem': line_no, 'motivo': 'matriculas_nao_lista'}); continue
            birth = parse_date(d.get('nascimento'))
            for mat_idx, mat in enumerate(mats):
                if not isinstance(mat, dict):
                    errors.append({'linha_origem': line_no, 'motivo': 'matricula_nao_objeto'}); continue
                school, cls = mat.get('escola') or {}, mat.get('turma') or {}
                if not isinstance(school, dict): school = {}
                if not isinstance(cls, dict): cls = {}
                presencas = mat.get('presencas') or []
                if not isinstance(presencas, list):
                    errors.append({'linha_origem': line_no, 'motivo': 'presencas_nao_lista'}); continue
                for p_idx, p in enumerate(presencas):
                    if not isinstance(p, dict):
                        errors.append({'linha_origem': line_no, 'motivo': 'presenca_nao_objeto'}); continue
                    rows.append({'rede':'B','codigo_inep':school.get('inep'),'escola':school.get('nome'),
                        'turma':cls.get('codigo') or cls.get('serie'),'turno':cls.get('turno'),
                        'identificador_origem':d.get('codigo_aluno') or (d.get('_id') or {}).get('$oid'),
                        'nome_aluno':d.get('nome'),'data_nascimento':birth,'data':parse_date(p.get('data')),
                        'status':p.get('status'),'inicio_matricula':parse_date(mat.get('inicio')),
                        'fim_matricula':parse_date(mat.get('fim')),'arquivo_origem':Path(path).name,
                        'linha_origem':line_no,'matricula_idx':mat_idx,'presenca_idx':p_idx})
    out = pd.DataFrame(rows)
    if not out.empty: out['codigo_inep'] = out['codigo_inep'].map(normalize_inep)
    return out, pd.DataFrame(errors), {'documentos_lidos':docs_read}

path_b = DADOS/'rede_b'/'alunos_rede_b.ndjson'
if path_b.exists():
    rede_b_raw, erros_b, stats_b = extract_b(path_b)
    safe_csv(rede_b_raw, BRONZE/'rede_b_frequencia_extraida.csv')
    safe_csv(erros_b, RESULTADOS/'erros_leitura_rede_b.csv')
    print(stats_b, '| presenças expandidas:', len(rede_b_raw), '| erros estruturais:', len(erros_b))
    display(rede_b_raw.head())
else:
    rede_b_raw = pd.DataFrame(); erros_b = pd.DataFrame(); print('NDJSON Rede B ainda não enviado.')

Saving alunos_rede_b.ndjson to alunos_rede_b.ndjson
Substituído alunos_rede_b.ndjson (1,132,120 bytes)
{'documentos_lidos': 311} | presenças expandidas: 19680 | erros estruturais: 0


,rede,codigo_inep,escola,turma,turno,identificador_origem,nome_aluno,data_nascimento,data,status,inicio_matricula,fim_matricula,arquivo_origem,linha_origem,matricula_idx,presenca_idx
0,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,Raí Araújo Cardoso,2011-11-26,2026-08-03,1,2026-02-02,None,alunos_rede_b.ndjson,1,0,0
1,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,Raí Araújo Cardoso,2011-11-26,2026-08-04,1,2026-02-02,None,alunos_rede_b.ndjson,1,0,1
2,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,Raí Araújo Cardoso,2011-11-26,2026-08-05,0,2026-02-02,None,alunos_rede_b.ndjson,1,0,2
3,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,Raí Araújo Cardoso,2011-11-26,2026-08-06,1,2026-02-02,None,alunos_rede_b.ndjson,1,0,3
4,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,Raí Araújo Cardoso,2011-11-26,2026-08-07,1,2026-02-02,None,alunos_rede_b.ndjson,1,0,4


## 5. Rede C — CSVs por escola/mês
Envie os arquivos em lotes. A célula substitui arquivos de mesmo nome canônico. Depois, a leitura verifica cada arquivo individualmente e registra problemas de cabeçalho/leitura no inventário.

In [9]:
uploaded = files.upload()
# Aceita os CSVs individualmente ou o ZIP com os 12 arquivos.
for upload_name, content in uploaded.items():
    suffix = Path(upload_name).suffix.lower()
    if suffix == '.zip':
        with zipfile.ZipFile(io.BytesIO(content)) as zf:
            csv_members = [m for m in zf.namelist() if m.lower().endswith('.csv') and not Path(m).name.startswith('.')]
            for member in csv_members:
                canonical = Path(member).name
                target = DADOS/'rede_c'/canonical
                target.write_bytes(zf.read(member))
                print('Salvo/substituído:', target.name)
    elif suffix == '.csv':
        save_uploads({upload_name: content}, DADOS/'rede_c', {'.csv'})
print('CSV presentes na pasta Rede C:', len(list((DADOS/'rede_c').glob('*.csv'))))

ALIASES_C = {
 'cod_inep':'codigo_inep','codigo_inep':'codigo_inep','inep':'codigo_inep','cod_inep_escola':'codigo_inep',
 'escola':'escola','nome_escola':'escola','turma':'turma','codigo_turma':'turma','turno':'turno',
 'matricula':'identificador_origem','matr':'identificador_origem','numero_matricula':'identificador_origem','numero_de_matricula':'identificador_origem',
 'nome_do_aluno':'nome_aluno','nome_aluno':'nome_aluno','nome':'nome_aluno','aluno':'nome_aluno',
 'data_nascimento':'data_nascimento','nascimento':'data_nascimento','dt_nasc':'data_nascimento','data':'data','data_aula':'data','dt_aula':'data',
 'situacao':'status','situacao_frequencia':'status','frequencia':'status','freq':'status','presenca':'status'
}

def read_csv_flexible(path):
    errors=[]
    for enc in ('utf-8-sig','utf-8','cp1252','latin-1'):
        for sep in (None, ';', ',','\t'):
            try:
                df = pd.read_csv(path, sep=sep, engine='python', dtype=str, encoding=enc, on_bad_lines='error')
                if len(df.columns) > 1: return df
            except Exception as e: errors.append(f'{enc}/{sep}: {e}')
    raise ValueError('Não foi possível ler o CSV. Tentativas: ' + ' | '.join(errors[-3:]))

def extract_c(directory):
    paths = sorted(Path(directory).glob('*.csv'))
    frames, report, errs = [], [], []
    required = ['codigo_inep','escola','turma','identificador_origem','nome_aluno','data_nascimento','data','status']
    for path in paths:
        try:
            df = read_csv_flexible(path)
            original_cols = list(df.columns)
            df.columns = [ALIASES_C.get(header_key(c), header_key(c)) for c in df.columns]
            # Caso existam nomes de colunas duplicados após normalização, não ocultar o problema.
            if df.columns.duplicated().any():
                raise ValueError(f'Colunas duplicadas após normalização: {df.columns[df.columns.duplicated()].tolist()}')
            missing = [c for c in required if c not in df.columns]
            if missing:
                report.append({'arquivo':path.name,'linhas_lidas':len(df),'status':'erro_colunas','detalhe':str(missing),'cabecalho_original':str(original_cols)})
                errs.append({'arquivo':path.name,'motivo':'colunas_obrigatorias_ausentes','detalhe':str(missing)})
                continue
            df['rede']='C'; df['codigo_inep']=df['codigo_inep'].map(normalize_inep)
            df['inicio_matricula']=None; df['fim_matricula']=None
            df['arquivo_origem']=path.name; df['linha_origem']=range(2, len(df)+2)
            frames.append(df)
            report.append({'arquivo':path.name,'linhas_lidas':len(df),'status':'ok','detalhe':'','cabecalho_original':str(original_cols)})
        except Exception as e:
            report.append({'arquivo':path.name,'linhas_lidas':0,'status':'erro_leitura','detalhe':str(e),'cabecalho_original':''})
            errs.append({'arquivo':path.name,'motivo':'erro_leitura','detalhe':str(e)})
    combined = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
    return combined, pd.DataFrame(report), pd.DataFrame(errs)

rede_c_raw, inventario_c, erros_c = extract_c(DADOS/'rede_c')
safe_csv(inventario_c, RESULTADOS/'inventario_rede_c.csv')
safe_csv(erros_c, RESULTADOS/'erros_leitura_rede_c.csv')
print('CSV encontrados:', len(list((DADOS/'rede_c').glob('*.csv'))), '/ 12 esperados')
print('Arquivos OK:', int(inventario_c.status.eq('ok').sum()) if not inventario_c.empty else 0)
print('Linhas extraídas:', len(rede_c_raw))
display(inventario_c); display(rede_c_raw.head())

Saving frequencia_5119_outubro_2026.csv to frequencia_5119_outubro_2026.csv
Saving frequencia_5119_agosto_2026.csv to frequencia_5119_agosto_2026.csv
Saving frequencia_8514_setembro_2026.csv to frequencia_8514_setembro_2026.csv
Saving frequencia_8954_agosto_2026.csv to frequencia_8954_agosto_2026.csv
Substituído frequencia_5119_outubro_2026.csv (265,610 bytes)
Substituído frequencia_5119_agosto_2026.csv (265,608 bytes)
Substituído frequencia_8514_setembro_2026.csv (277,753 bytes)
Substituído frequencia_8954_agosto_2026.csv (262,290 bytes)
CSV presentes na pasta Rede C: 12
CSV encontrados: 12 / 12 esperados
Arquivos OK: 12
Linhas extraídas: 22623


,arquivo,linhas_lidas,status,detalhe,cabecalho_original
0,frequencia_5119_agosto_2026.csv,1808,ok,,"['Cod_INEP', 'Escola', 'Turma', 'Turno', 'Matr..."
1,frequencia_5119_outubro_2026.csv,1809,ok,,"['INEP', 'ESCOLA', 'TURMA', 'TURNO', 'MATR', '..."
2,frequencia_5119_setembro_2026.csv,1810,ok,,"['codigo inep', 'nome escola', 'turma', 'turno..."
3,frequencia_5906_agosto_2026.csv,1733,ok,,"['Cod_INEP', 'Escola', 'Turma', 'Turno', 'Matr..."
4,frequencia_5906_outubro_2026.csv,1705,ok,,"['INEP', 'ESCOLA', 'TURMA', 'TURNO', 'MATR', '..."
5,frequencia_5906_setembro_2026.csv,1734,ok,,"['codigo inep', 'nome escola', 'turma', 'turno..."
6,frequencia_8514_agosto_2026.csv,1912,ok,,"['Cod_INEP', 'Escola', 'Turma', 'Turno', 'Matr..."
7,frequencia_8514_outubro_2026.csv,2002,ok,,"['INEP', 'ESCOLA', 'TURMA', 'TURNO', 'MATR', '..."
8,frequencia_8514_setembro_2026.csv,1916,ok,,"['codigo inep', 'nome escola', 'turma', 'turno..."
9,frequencia_8954_agosto_2026.csv,1913,ok,,"['Cod_INEP', 'Escola', 'Turma', 'Turno', 'Matr..."


,codigo_inep,escola,turma,turno,identificador_origem,nome_aluno,data_nascimento,data,status,rede,inicio_matricula,fim_matricula,arquivo_origem,linha_origem
0,99675119,Escola Municipal de Ensino Fundamental Prof. S...,8º ANO A,Tarde,2026578131,Heitor Gomes Queiroz,2013-07-06,2026-08-07,J,C,None,None,frequencia_5119_agosto_2026.csv,2
1,99675119,Escola Municipal de Ensino Fundamental Prof. S...,7º ANO A,Tarde,2026686770,Mateus Freitas Silva,2012-02-15,2026-08-25,P,C,None,None,frequencia_5119_agosto_2026.csv,3
2,99675119,Escola Municipal de Ensino Fundamental Prof. S...,6º ANO A,Tarde,2026450164,Letícia Silva Tavares,2012-02-07,2026-08-14,P,C,None,None,frequencia_5119_agosto_2026.csv,4
3,99675119,Escola Municipal de Ensino Fundamental Prof. S...,7º ANO A,Tarde,2026476823,Gabriel Siqueira Souza,2011-02-03,2026-08-31,P,C,None,None,frequencia_5119_agosto_2026.csv,5
4,99675119,Escola Municipal de Ensino Fundamental Prof. S...,8º ANO A,Tarde,2026788214,Bernardo Melo Souza,2015-09-18,2026-08-20,P,C,None,None,frequencia_5119_agosto_2026.csv,6


## 6. Transformação comum e regras R1–R7
Esta etapa registra todos os motivos de descarte por registro. Um mesmo registro pode ter mais de um motivo, por isso o total de registros descartados é calculado separadamente (uma única vez por linha de origem).

In [10]:
def extract_a(tables):
    if not tables: return pd.DataFrame()
    # Normaliza nomes de colunas sem assumir nomes de apresentação em merges.
    aluno = tables['aluno'].rename(columns={'id':'aluno_id','nome':'nome_aluno'}).copy()
    escola = tables['escola'].rename(columns={'id':'escola_id','nome':'nome_escola'}).copy()
    turma = tables['turma'].rename(columns={'codigo':'turma'}).copy()
    matricula = tables['matricula'].copy()
    freq = tables['frequencia'].copy()
    for df, cols in [(aluno,['aluno_id']), (turma,['id','escola_id']), (escola,['escola_id']), (matricula,['aluno_id','turma_id']), (freq,['aluno_id','turma_id'])]:
        missing = set(cols) - set(df.columns)
        if missing: raise ValueError(f'Colunas esperadas ausentes na Rede A: {missing}')
    # Converte presente preservando nulo como chamada não registrada.
    freq['presente_raw'] = freq['presente']
    f = freq.merge(aluno, on='aluno_id', how='left', indicator='_aluno_merge')
    f = f.merge(matricula[['aluno_id','turma_id','data_matricula']], on=['aluno_id','turma_id'], how='left', indicator='_matricula_merge')
    f = f.merge(turma[['id','escola_id','turma','serie','turno']].rename(columns={'id':'turma_id'}), on='turma_id', how='left', indicator='_turma_merge')
    f = f.merge(escola[['escola_id','codigo_inep','nome_escola']], on='escola_id', how='left', indicator='_escola_merge')
    out = pd.DataFrame({
      'rede':'A','codigo_inep':f['codigo_inep'],'escola':f['nome_escola'],'turma':f['turma'].where(f['turma'].notna(),f['serie']),
      'turno':f['turno'],'identificador_origem':f['aluno_id'],'nome_aluno':f['nome_aluno'],
      'data_nascimento':f['data_nascimento'],'data':f['data_aula'],'status':f['presente_raw'],
      'inicio_matricula':f['data_matricula'],'fim_matricula':None,
      'escola_existente':f['_escola_merge'].eq('both'),'aluno_existente':f['_aluno_merge'].eq('both'),
      'turma_existente':f['_turma_merge'].eq('both'),'matricula_existente':f['_matricula_merge'].eq('both'),
      'arquivo_origem':'rede_a_dump.sql','linha_origem':range(1,len(f)+1)})
    return out

# Para B/C, a escola é identificada pelos dados da própria fonte; para A, a existência é validada por joins no dump.
def standardize_validate(df, calendar_days, network):
    if df.empty:
        return pd.DataFrame(), pd.DataFrame(), {'rede':network,'lidos':0,'validos':0,'descartados':0,'duplicados_removidos':0,'cadastros_unificados':0}
    x=df.copy().reset_index(drop=True)
    for col in ['rede','codigo_inep','escola','turma','identificador_origem','nome_aluno','data_nascimento','data','status','inicio_matricula','fim_matricula']:
        if col not in x: x[col]=None
    x['rede']=network
    x['codigo_inep']=x['codigo_inep'].map(normalize_inep)
    x['data']=x['data'].map(parse_date)
    x['data_nascimento']=x['data_nascimento'].map(parse_date)
    x['inicio_matricula']=x['inicio_matricula'].map(parse_date)
    x['fim_matricula']=x['fim_matricula'].map(parse_date)
    x['nome_norm']=x['nome_aluno'].map(norm_text)
    x['turma_norm']=x['turma'].map(norm_text)
    x['escola_norm']=x['escola'].map(norm_text)
    x['presencas']=x['status'].map(parse_bool_status)
    x['mes']=x['data'].map(lambda d: d[:7] if d else None)
    x['chave_aluno']=x['nome_norm']+'|'+x['data_nascimento'].fillna('')
    x['identificador_aluno']=x['chave_aluno'].map(lambda key: hashlib.sha256(f'{network}|{key}'.encode()).hexdigest()[:16] if key and not key.endswith('|') else None)
    # Validar existência real de escola/turma/aluno na Rede A pelos joins; B/C usam escola/turma declaradas na fonte.
    school_exists = x['escola_existente'].fillna(False).astype(bool) if 'escola_existente' in x else (x['codigo_inep'].notna() & x['escola_norm'].ne(''))
    class_exists = (x['turma_existente'].fillna(False).astype(bool) if 'turma_existente' in x else pd.Series(True, index=x.index)) & x['turma_norm'].ne('')
    student_exists = (x['aluno_existente'].fillna(False).astype(bool) if 'aluno_existente' in x else pd.Series(True, index=x.index)) & x['nome_norm'].ne('') & x['data_nascimento'].notna()
    date_ok = x['data'].notna() & x['data'].between(DATA_INICIO, DATA_FIM) & x['data'].isin(calendar_days)
    status_ok = x['presencas'].notna()
    inep_ok = x['codigo_inep'].map(lambda v: bool(re.fullmatch(r'\d{8}', v or '')))
    valid_matricula = x['inicio_matricula'].isna() | (x['data'] >= x['inicio_matricula'])
    valid_matricula &= x['fim_matricula'].isna() | (x['data'] <= x['fim_matricula'])
    checks = {
      'R1_data_fora_periodo_ou_nao_letiva': date_ok,
      'R2_data_invalida': x['data'].notna(),
      'R2_aluno_nao_identificado': student_exists,
      'R2_turma_nao_identificada': class_exists,
      'R2_escola_inexistente_ou_nao_identificada': school_exists,
      'R2_situacao_frequencia_ausente_ou_desconhecida': status_ok,
      'R6_fora_vigencia_matricula': valid_matricula,
      'R7_codigo_inep_invalido': inep_ok,
    }
    # Registro com data inválida recebe R2 também; R1 registra fora do período/calendário apenas se a data existir.
    reasons=[]
    for reason, ok in checks.items():
        bad = ~ok.fillna(False)
        # R1 só é aplicável a datas interpretáveis; data inválida tem motivo próprio R2.
        if reason == 'R1_data_fora_periodo_ou_nao_letiva': bad &= x['data'].notna()
        if bad.any():
            cols = [c for c in ['arquivo_origem','linha_origem'] if c in x.columns]
            bad_rows = x.loc[bad, cols].copy()
            bad_rows['indice_registro'] = bad_rows.index.astype(int)
            bad_rows['rede'] = network
            bad_rows['motivo'] = reason
            reasons.extend(bad_rows[['rede','indice_registro','arquivo_origem','linha_origem','motivo']].to_dict('records') if {'arquivo_origem','linha_origem'}.issubset(bad_rows.columns) else [dict(rede=network, indice_registro=int(i), arquivo_origem=x.at[i,'arquivo_origem'] if 'arquivo_origem' in x else '', linha_origem=x.at[i,'linha_origem'] if 'linha_origem' in x else None, motivo=reason) for i in bad_rows.index])
    invalid_idx=set(r['indice_registro'] for r in reasons if r['motivo'] != 'R5_duplicidade_removida')
    # Se turma/escola declarada existe apenas por estar preenchida nas fontes B/C, exigir também nome e INEP válidos.
    valid_mask = pd.Series(True,index=x.index)
    for ok in checks.values(): valid_mask &= ok.fillna(False)
    valid = x.loc[valid_mask].copy()
    # Unificar identidade por nome normalizado+nascimento dentro da rede. Contar aliases de cadastro distintos.
    identity_counts = valid.groupby('chave_aluno')['identificador_origem'].nunique(dropna=True)
    unified = int((identity_counts[identity_counts > 1] - 1).sum())
    # Uma linha por aluno, escola, turma e data. Keep deterministic first row; order by source file/line.
    sort_cols=[c for c in ['arquivo_origem','linha_origem','identificador_origem'] if c in valid.columns]
    if sort_cols: valid=valid.sort_values(sort_cols, kind='stable')
    dupe_key=['rede','chave_aluno','codigo_inep','turma_norm','data']
    dup_mask=valid.duplicated(dupe_key, keep='first')
    duplicate_count=int(dup_mask.sum())
    for idx in valid.index[dup_mask]:
        row=valid.loc[idx]
        reasons.append({'rede':network,'indice_registro':int(idx),'arquivo_origem':row.get('arquivo_origem',''),'linha_origem':row.get('linha_origem',None),'motivo':'R5_duplicidade_removida'})
    valid=valid.loc[~dup_mask].copy()
    # Campos finais analíticos, sem identificadores sensíveis desnecessários.
    final_cols=['rede','codigo_inep','escola','turma','turno','identificador_origem','identificador_aluno','nome_aluno','data_nascimento','data','mes','status','presencas','inicio_matricula','fim_matricula']
    for c in final_cols:
        if c not in valid: valid[c]=None
    valid=valid[final_cols].copy()
    valid['presencas']=valid['presencas'].astype('int64')
    reason_df=pd.DataFrame(reasons, columns=['rede','indice_registro','arquivo_origem','linha_origem','motivo'])
    quality={'rede':network,'lidos':len(x),'validos':len(valid),'descartados':len(invalid_idx)+duplicate_count,'duplicados_removidos':duplicate_count,'cadastros_unificados':unified}
    return valid, reason_df, quality

# Monta as fontes normalizadas
fontes = {}
if tabelas_a:
    fontes['A'] = extract_a(tabelas_a)
if not rede_b_raw.empty: fontes['B'] = rede_b_raw.copy()
if not rede_c_raw.empty: fontes['C'] = rede_c_raw.copy()
if not fontes:
    raise RuntimeError('Nenhuma fonte carregada. Envie os arquivos das redes A, B e C.')

silver_por_rede, motivos_por_rede, resumo_qualidade = {}, [], []
for rede, df in fontes.items():
    silver, motivos, resumo = standardize_validate(df, dias_letivos, rede)
    silver_por_rede[rede] = silver
    motivos_por_rede.append(motivos)
    resumo_qualidade.append(resumo)
    safe_csv(silver, SILVER/f'frequencia_rede_{rede.lower()}.csv')
    print(f"Rede {rede}: lidos={resumo['lidos']:,}; válidos={resumo['validos']:,}; descartados únicos={resumo['descartados']:,}; duplicados removidos={resumo['duplicados_removidos']:,}; cadastros unificados={resumo['cadastros_unificados']:,}")
    display(silver.head(3))

motivos_df = pd.concat([d for d in motivos_por_rede if not d.empty], ignore_index=True) if any(not d.empty for d in motivos_por_rede) else pd.DataFrame(columns=['rede','indice_registro','arquivo_origem','linha_origem','motivo'])
resumo_df = pd.DataFrame(resumo_qualidade)
safe_csv(motivos_df, RESULTADOS/'motivos_descarte.csv')
safe_csv(resumo_df, RESULTADOS/'resumo_qualidade.csv')
display(resumo_df); display(motivos_df['motivo'].value_counts().rename_axis('motivo').reset_index(name='quantidade'))

Rede A: lidos=26,177; válidos=23,314; descartados únicos=2,863; duplicados removidos=269; cadastros unificados=10


,rede,codigo_inep,escola,turma,turno,identificador_origem,identificador_aluno,nome_aluno,data_nascimento,data,mes,status,presencas,inicio_matricula,fim_matricula
0,A,99465873,Escola Municipal Cônego Luís Barreto,5A,Tarde,238,656113f2226b6d18,Bianca Rodrigues Nascimento,2011-11-03,2026-09-28,2026-09,t,1,2026-02-02,None
1,A,99465873,Escola Municipal Cônego Luís Barreto,7A,Tarde,280,486f7990ab315f96,Ícaro Batista Ferreira,2013-05-11,2026-10-08,2026-10,t,1,2026-02-02,None
2,A,99018898,Escola Municipal de Ensino Fundamental Profª M...,5A,Tarde,310,2ac5ed6bb40900f2,Bernardo Batista Barros,2013-04-02,2026-09-22,2026-09,t,1,2026-02-02,None


Rede B: lidos=19,680; válidos=19,191; descartados únicos=489; duplicados removidos=251; cadastros unificados=0


,rede,codigo_inep,escola,turma,turno,identificador_origem,identificador_aluno,nome_aluno,data_nascimento,data,mes,status,presencas,inicio_matricula,fim_matricula
0,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,e195d01cf80bea64,Raí Araújo Cardoso,2011-11-26,2026-08-03,2026-08,1,1,2026-02-02,None
1,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,e195d01cf80bea64,Raí Araújo Cardoso,2011-11-26,2026-08-04,2026-08,1,1,2026-02-02,None
2,B,99473471,ESCOLA MUNICIPAL INTEGRAL DR. ULISSES PONTES,T3471-6A,Tarde,RB-00145,e195d01cf80bea64,Raí Araújo Cardoso,2011-11-26,2026-08-05,2026-08,0,0,2026-02-02,None


Rede C: lidos=22,623; válidos=16,477; descartados únicos=6,146; duplicados removidos=451; cadastros unificados=0


,rede,codigo_inep,escola,turma,turno,identificador_origem,identificador_aluno,nome_aluno,data_nascimento,data,mes,status,presencas,inicio_matricula,fim_matricula
0,C,99675119,Escola Municipal de Ensino Fundamental Prof. S...,8º ANO A,Tarde,2026578131,520300e21abc937f,Heitor Gomes Queiroz,2013-07-06,2026-08-07,2026-08,J,0,None,None
1,C,99675119,Escola Municipal de Ensino Fundamental Prof. S...,7º ANO A,Tarde,2026686770,ebb46555c17d71ef,Mateus Freitas Silva,2012-02-15,2026-08-25,2026-08,P,1,None,None
2,C,99675119,Escola Municipal de Ensino Fundamental Prof. S...,6º ANO A,Tarde,2026450164,af807d67f21ab280,Letícia Silva Tavares,2012-02-07,2026-08-14,2026-08,P,1,None,None


,rede,lidos,validos,descartados,duplicados_removidos,cadastros_unificados
0,A,26177,23314,2863,269,10
1,B,19680,19191,489,251,0
2,C,22623,16477,6146,451,0


,motivo,quantidade
0,R2_escola_inexistente_ou_nao_identificada,7698
1,R7_codigo_inep_invalido,7698
2,R5_duplicidade_removida,971
3,R1_data_fora_periodo_ou_nao_letiva,424
4,R2_situacao_frequencia_ausente_ou_desconhecida,334
5,R2_aluno_nao_identificado,44
6,R6_fora_vigencia_matricula,40
7,R2_data_invalida,34
8,R2_turma_nao_identificada,3


## 7. Checagem de completude antes dos indicadores finais
**Não publique indicadores como finais** se não estiverem presentes Rede A, Rede B e os 12 CSVs válidos da Rede C. É permitido tratar fontes separadamente, mas a consolidação completa exige todas as fontes.

In [11]:
arquivos_c = sorted((DADOS/'rede_c').glob('*.csv'))
rede_c_ok = int(inventario_c.status.eq('ok').sum()) if not inventario_c.empty else 0
fontes_faltantes = sorted({'A','B','C'} - set(fontes))
completo = not fontes_faltantes and len(arquivos_c) == 12 and rede_c_ok == 12
print('Redes presentes:', sorted(fontes.keys()))
print(f'CSVs Rede C: {len(arquivos_c)} arquivos encontrados; {rede_c_ok} processados com sucesso; esperado: 12')
if not completo:
    print('STATUS: PARCIAL — não tratar os indicadores como entrega final.')
    print('Redes faltantes:', fontes_faltantes)
    print('Arquivos Rede C faltantes/ inválidos:', max(0, 12-rede_c_ok))
else:
    print('STATUS: fontes completas para consolidação.')

Redes presentes: ['A', 'B', 'C']
CSVs Rede C: 12 arquivos encontrados; 12 processados com sucesso; esperado: 12
STATUS: fontes completas para consolidação.


## 8. Gerar os três resultados obrigatórios
Esta célula bloqueia a exportação final se as fontes não estiverem completas. O layout das três saídas segue exatamente as colunas obrigatórias do enunciado.

In [12]:
if not completo:
    raise RuntimeError('Fontes incompletas: não gere/entregue resultados finais. Complete Rede A, Rede B e os 12 CSVs Rede C e execute novamente as seções 5–8.')

all_silver = pd.concat([silver_por_rede[r] for r in ['A','B','C']], ignore_index=True)
if all_silver.empty: raise ValueError('Nenhum registro válido após tratamento.')

# Não agregamos percentuais; somamos numeradores e denominadores.
def aggregate_frequency(df, keys):
    out = df.groupby(keys, dropna=False).agg(
        registros_validos=('presencas','size'),
        presencas=('presencas','sum')
    ).reset_index()
    out['frequencia_pct'] = (100.0 * out['presencas'] / out['registros_validos']).round(2)
    return out

escola_mes = aggregate_frequency(all_silver, ['rede','codigo_inep','mes'])
turma_mes = aggregate_frequency(all_silver, ['rede','codigo_inep','turma','mes'])
aluno_periodo = all_silver.groupby(['rede','identificador_aluno'], dropna=False).agg(
    nome_aluno=('nome_aluno','first'),
    data_nascimento=('data_nascimento','first'),
    registros_validos=('presencas','size'),
    presencas=('presencas','sum')
).reset_index()
aluno_periodo['frequencia_pct'] = (100.0*aluno_periodo['presencas']/aluno_periodo['registros_validos']).round(2)
abaixo_75 = aluno_periodo.loc[aluno_periodo['frequencia_pct'] < 75.00].copy()

# Ordem exata exigida pelo documento.
escola_mes = escola_mes[['rede','codigo_inep','mes','registros_validos','presencas','frequencia_pct']].sort_values(['rede','codigo_inep','mes'])
turma_mes = turma_mes[['rede','codigo_inep','turma','mes','registros_validos','presencas','frequencia_pct']].sort_values(['rede','codigo_inep','turma','mes'])
abaixo_75 = abaixo_75[['rede','identificador_aluno','nome_aluno','data_nascimento','registros_validos','presencas','frequencia_pct']].sort_values(['rede','identificador_aluno'])

safe_csv(all_silver, SILVER/'frequencia_todas_redes.csv')
escola_mes.to_csv(RESULTADOS/'frequencia_escola_mes.csv', index=False, encoding='utf-8', float_format='%.2f', na_rep='')
turma_mes.to_csv(RESULTADOS/'frequencia_turma_mes.csv', index=False, encoding='utf-8', float_format='%.2f', na_rep='')
abaixo_75.to_csv(RESULTADOS/'alunos_abaixo_75.csv', index=False, encoding='utf-8', float_format='%.2f', na_rep='')

print('Resultados exportados:')
for name, df, expected in [
 ('frequencia_escola_mes.csv', escola_mes, ['rede','codigo_inep','mes','registros_validos','presencas','frequencia_pct']),
 ('frequencia_turma_mes.csv', turma_mes, ['rede','codigo_inep','turma','mes','registros_validos','presencas','frequencia_pct']),
 ('alunos_abaixo_75.csv', abaixo_75, ['rede','identificador_aluno','nome_aluno','data_nascimento','registros_validos','presencas','frequencia_pct'])]:
    p=RESULTADOS/name
    check=pd.read_csv(p, dtype=str)
    assert list(check.columns)==expected, f'Layout incorreto: {name}'
    print(name, '| linhas:',len(check),'| colunas conferidas:',list(check.columns))
    display(check.head())

Resultados exportados:
frequencia_escola_mes.csv | linhas: 36 | colunas conferidas: ['rede', 'codigo_inep', 'mes', 'registros_validos', 'presencas', 'frequencia_pct']


,rede,codigo_inep,mes,registros_validos,presencas,frequencia_pct
0,A,99018898,2026-08,1405,1160,82.56
1,A,99018898,2026-09,1403,1140,81.25
2,A,99018898,2026-10,1402,1081,77.10
3,A,99158813,2026-08,1632,1463,89.64
4,A,99158813,2026-09,1636,1442,88.14


frequencia_turma_mes.csv | linhas: 117 | colunas conferidas: ['rede', 'codigo_inep', 'turma', 'mes', 'registros_validos', 'presencas', 'frequencia_pct']


,rede,codigo_inep,turma,mes,registros_validos,presencas,frequencia_pct
0,A,99018898,5A,2026-08,461,373,80.91
1,A,99018898,5A,2026-09,460,368,80.00
2,A,99018898,5A,2026-10,460,337,73.26
3,A,99018898,6A,2026-08,483,422,87.37
4,A,99018898,6A,2026-09,481,398,82.74


alunos_abaixo_75.csv | linhas: 165 | colunas conferidas: ['rede', 'identificador_aluno', 'nome_aluno', 'data_nascimento', 'registros_validos', 'presencas', 'frequencia_pct']


,rede,identificador_aluno,nome_aluno,data_nascimento,registros_validos,presencas,frequencia_pct
0,A,0849c4fef826274e,José Santos Macêdo,2011-08-10,63,44,69.84
1,A,0ea7ca0ec301c451,Raí Leão Vasconcelos,2013-08-29,63,38,60.32
2,A,14f83f5b7f75b3f9,Júlia Costa Cavalcanti,2014-01-17,63,25,39.68
3,A,1e643d10bb95a53f,Ingrid Macêdo Rocha,2011-04-02,63,44,69.84
4,A,20729af566d18e3d,Arthur Ferreira Martins,2013-02-27,63,47,74.60


## 9. Relatório de qualidade final
O relatório diferencia registros lidos, registros válidos, registros descartados únicos, motivos (que podem se sobrepor), duplicidades removidas e cadastros unificados. Também registra arquivos inválidos/incompletos. A contagem de motivos pode exceder o número de registros descartados porque uma linha pode ter múltiplos problemas.

In [13]:
# Atualiza o resumo para ficar junto dos resultados entregáveis.
resumo_df = pd.DataFrame(resumo_qualidade)
linhas_relatorio = [
    '# Relatório de qualidade — Case Motriz', '',
    f'- Período avaliado: {DATA_INICIO} a {DATA_FIM}.',
    f'- Dias letivos no calendário: {len(dias_letivos)}.',
    f'- CSVs Rede C encontrados: {len(arquivos_c)}/12; processados com sucesso: {rede_c_ok}/12.',
    '- Motivos de descarte não são mutuamente exclusivos; o total de registros descartados conta cada registro uma vez.', '',
    '## Resumo por rede', '',
    '| Rede | Lidos | Válidos | Descartados únicos | Duplicidades removidas | Cadastros unificados |',
    '|---|---:|---:|---:|---:|---:|'
]
for _, r in resumo_df.iterrows():
    linhas_relatorio.append(f"| {r['rede']} | {r['lidos']} | {r['validos']} | {r['descartados']} | {r['duplicados_removidos']} | {r['cadastros_unificados']} |")
linhas_relatorio += ['', '## Motivos identificados', '']
if not motivos_df.empty:
    for (rede, motivo), grp in motivos_df.groupby(['rede','motivo']):
        linhas_relatorio.append(f'- Rede {rede} — `{motivo}`: {len(grp)} ocorrência(s).')
else:
    linhas_relatorio.append('- Nenhum motivo de descarte registrado.')
if not erros_b.empty:
    linhas_relatorio += ['', '## Erros estruturais NDJSON', f'- Linhas/documentos com erro: {len(erros_b)}.']
if not erros_c.empty:
    linhas_relatorio += ['', '## Problemas de leitura Rede C', f'- Arquivos/ocorrências com erro: {len(erros_c)}.']
linhas_relatorio += ['', '## Decisões e limitações',
    '- Identidade do aluno: nome normalizado + data de nascimento, limitada à própria rede.',
    '- Deduplicação: rede + identidade do aluno + INEP + turma + data da aula.',
    '- Faltas justificadas são denominador sem numerador; situações desconhecidas são descartadas.',
    '- Para Rede B/C, a existência de escola/turma é inferida dos campos declarados na própria fonte, pois o enunciado não fornece cadastro mestre externo dessas redes.',
    '- Os resultados finais só são produzidos quando todas as fontes obrigatórias estão presentes e os 12 CSVs da Rede C são lidos com sucesso.'
]
(RESULTADOS/'relatorio_qualidade.md').write_text('\n'.join(linhas_relatorio)+'\n', encoding='utf-8')
safe_csv(resumo_df, RESULTADOS/'resumo_qualidade.csv')
safe_csv(motivos_df, RESULTADOS/'motivos_descarte.csv')
print((RESULTADOS/'relatorio_qualidade.md').read_text())

# Relatório de qualidade — Case Motriz

- Período avaliado: 2026-08-03 a 2026-10-30.
- Dias letivos no calendário: 63.
- CSVs Rede C encontrados: 12/12; processados com sucesso: 12/12.
- Motivos de descarte não são mutuamente exclusivos; o total de registros descartados conta cada registro uma vez.

## Resumo por rede

| Rede | Lidos | Válidos | Descartados únicos | Duplicidades removidas | Cadastros unificados |
|---|---:|---:|---:|---:|---:|
| A | 26177 | 23314 | 2863 | 269 | 10 |
| B | 19680 | 19191 | 489 | 251 | 0 |
| C | 22623 | 16477 | 6146 | 451 | 0 |

## Motivos identificados

- Rede A — `R1_data_fora_periodo_ou_nao_letiva`: 226 ocorrência(s).
- Rede A — `R2_aluno_nao_identificado`: 40 ocorrência(s).
- Rede A — `R2_escola_inexistente_ou_nao_identificada`: 2268 ocorrência(s).
- Rede A — `R2_situacao_frequencia_ausente_ou_desconhecida`: 60 ocorrência(s).
- Rede A — `R5_duplicidade_removida`: 269 ocorrência(s).
- Rede A — `R7_codigo_inep_invalido`: 2268 ocorrência(s).
- Rede B — `

## 10. Empacotar dados tratados para o Notebook 02
O pacote contém apenas Silver e resultados. Não inclui os arquivos brutos nem credenciais. A carga no BigQuery será feita em outro notebook.

In [14]:
pacote = shutil.make_archive(str(BASE/'dados_tratados_para_bigquery'), 'zip', root_dir=BASE, base_dir='silver')
print('Pacote criado:', pacote)
print('Silver:', [p.name for p in SILVER.glob('*.csv')])
print('Resultados:', [p.name for p in RESULTADOS.iterdir() if p.is_file()])
# Para baixar: files.download(pacote)

Pacote criado: /content/drive/MyDrive/motriz_case/dados_tratados_para_bigquery.zip
Silver: ['frequencia_rede_a.csv', 'frequencia_rede_b.csv', 'frequencia_rede_c.csv', 'frequencia_todas_redes.csv']
Resultados: ['erros_leitura_rede_b.csv', 'inventario_rede_c.csv', 'erros_leitura_rede_c.csv', 'motivos_descarte.csv', 'resumo_qualidade.csv', 'frequencia_escola_mes.csv', 'frequencia_turma_mes.csv', 'alunos_abaixo_75.csv', 'relatorio_qualidade.md']


## 11. Checklist final antes do BigQuery
- [ ] Calendário carregado e `dia_letivo = S` conferido.
- [ ] Dump Rede A extraído com as cinco tabelas esperadas.
- [ ] NDJSON Rede B lido; erros estruturais revisados.
- [ ] 12 CSVs Rede C presentes e processados sem erro de cabeçalho/leitura.
- [ ] Relatório de qualidade revisado e decisões justificadas no README.
- [ ] Layout exato dos três CSVs conferido automaticamente.
- [ ] Resultados recalculados depois de qualquer substituição de arquivo.

**Nota técnica:** a validação de existência de escola/turma na Rede A usa os relacionamentos presentes no dump. Nas Redes B e C, sem cadastro mestre independente fornecido no enunciado, a validação possível é estrutural (INEP, nome da escola e turma preenchidos); essa limitação deve ser explicada no README.